# DLC Spatial Analysis (Reward-Aligned)

Spatial analysis of DLC tracking data aligned to reward events.
Extends Phase 5's whole-session analyses with peri-reward windowing.

Outputs go to `stroke_orofacial_pipeline_outputs/dlc_analysis/spatial_reward_aligned/`
on MICROSCOPE.

In [ ]:
# ============================================================
# USER INPUTS — edit variables below, then run all cells.
# ============================================================

ANIMALS = ["PS46"]

# Dates to process. None = all dates in session range.
DATES = None

# Bodyparts and centering modes
BODYPARTS = ["tongue", "jaw", "nose"]
CENTER_MODES = ["nose", "spout_midpoint", "eye_midpoint"]

# Reward windows (ms): [pre_ms, post_ms]
REWARD_WINDOWS_MS = [(-500, 5000), (-100, 1000)]

# Trial groups
TRIAL_GROUPS = ["all", "L", "R"]

# Overwrite mode: "skip" or "force"
OVERWRITE = "skip"

In [ ]:
import datetime
import logging
from pathlib import Path

import numpy as np

from stroke_orofacial.animals import load_animals
from stroke_orofacial.config_loader import load_defaults
from stroke_orofacial.paths import PathResolver
from stroke_orofacial.dlc_spatial.reward_aligned import build_reward_aligned_analysis
from stroke_orofacial.dlc_spatial.heatmaps import plot_heatmap, save_heatmap_png

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [ ]:
resolver = PathResolver()
animals_config = load_animals()
defaults = load_defaults()
spatial_config = defaults["dlc_spatial"]
reward_config = defaults["dlc_spatial_reward_aligned"]
output_root = resolver.pipeline_outputs("dlc_analysis") / "spatial_reward_aligned"

for animal_id in ANIMALS:
    info = animals_config[animal_id]

    if DATES is not None:
        dates = DATES
    else:
        n_days = (info.session_end - info.session_start).days + 1
        dates = [info.session_start + datetime.timedelta(days=i) for i in range(n_days)]

    for center_mode in CENTER_MODES:
        for bodypart in BODYPARTS:
            for window_ms in REWARD_WINDOWS_MS:
                for trial_group in TRIAL_GROUPS:
                    n_ok, n_skip, n_err = 0, 0, 0

                    for date in dates:
                        date_str = date.strftime("%Y%m%d")
                        win_tag = f"{int(window_ms[0])}to{int(window_ms[1])}ms"
                        out_dir = (
                            output_root / f"{bodypart}_{center_mode}"
                            / f"trials-{trial_group}_{win_tag}" / animal_id
                        )
                        out_path = out_dir / f"heatmap_{date_str}.png"

                        if OVERWRITE == "skip" and out_path.exists():
                            n_skip += 1
                            continue

                        try:
                            result = build_reward_aligned_analysis(
                                animal_id, date_str, bodypart, center_mode,
                                trial_group, window_ms, resolver,
                                spatial_config, reward_config,
                            )
                            if result is None or result["histogram"] is None:
                                n_err += 1
                                continue

                            # Save heatmap PNG
                            from stroke_orofacial.dlc_spatial.pixel_bins import build_histogram
                            hist = result["histogram"]
                            import matplotlib.pyplot as plt
                            from matplotlib.colors import LogNorm

                            fig, ax = plt.subplots(figsize=(4.6, 4.4))
                            if hist["H_counts"].sum() > 0:
                                ax.imshow(
                                    hist["H_proportion"].T,
                                    extent=hist["extent"],
                                    origin="lower", aspect="auto",
                                    cmap="magma",
                                    norm=LogNorm(vmin=1e-5),
                                )
                            ax.set_title(
                                f"{animal_id} {bodypart} {date_str}\n"
                                f"{center_mode} | {trial_group} | {win_tag}\n"
                                f"rewards={result['n_rewards']}, frames={result['n_selected_frames']}"
                            )
                            ax.set_xlabel(f"{bodypart}_x")
                            ax.set_ylabel(f"{bodypart}_y")
                            out_path.parent.mkdir(parents=True, exist_ok=True)
                            fig.savefig(str(out_path), dpi=200, bbox_inches="tight")
                            plt.close(fig)

                            n_ok += 1
                        except Exception as e:
                            n_err += 1
                            logging.warning("%s/%s/%s/%s/%s: %s",
                                animal_id, date_str, bodypart, center_mode, trial_group, e)

                    tag = f"{bodypart}/{center_mode}/trials-{trial_group}/{int(window_ms[0])}to{int(window_ms[1])}ms"
                    if n_ok + n_skip + n_err > 0:
                        print(f"  {tag}: ok={n_ok} skip={n_skip} err={n_err}")